# Outlines CFG — Always-Valid Code and SQL

**Week 2 | Notebook 3 of 4**

**What you'll learn:**
- Context-free grammars in 10 minutes
- SQL grammar — generating valid SELECT/WHERE queries
- Python expression grammar
- Custom DSL grammar — defining your own language
- Comparing CFG output vs. raw code generation (syntax error rates)
- Integrating Outlines with vLLM for production serving

**Runtime:** ~45 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("02_outlines/03_cfg_codegen.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  02_outlines/03_cfg_codegen.ipynb
Task:      CFG-guided code/SQL generation
Calls:     ~15

With GPT-4o:       $0.15 USD
With GPT-4o-mini:  $0.01 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [2]:
import outlines
import torch
from outlines.types import CFG
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings

warnings.filterwarnings("ignore")

# CFG constraints require token-level masking, so they need a LOCAL model —
# hosted APIs (OpenAI etc.) only support JSON-schema constraints.
# First run downloads ~1 GB (Qwen2.5-0.5B); it's cached afterwards.
LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
hf_model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=torch.float32)
model = outlines.from_transformers(hf_model, tokenizer)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 2. What Are Context-Free Grammars? (Lark Format)

In [3]:
# A simple arithmetic grammar in Lark format
arithmetic_grammar = r"""
    start: expr
    expr: expr "+" term
        | expr "-" term
        | term
    term: term "*" factor
        | term "/" factor
        | factor
    factor: NUMBER
          | "(" expr ")"
    NUMBER: /[0-9]+(\.[0-9]+)?/
"""

result = model("Write a mathematical expression for compound interest:", CFG(arithmetic_grammar))

print(f"Generated expression: {result}")
print("✅ Always syntactically valid arithmetic")

W0921 07:22:38.414000 19272 Lib\site-packages\torch\_dynamo\convert_frame.py:2287] WON'T CONVERT apply_token_bitmask_inplace_kernel f:\paul\structured-llm-notebooks\.venv\Lib\site-packages\llguidance\torch.py line 26 
W0921 07:22:38.414000 19272 Lib\site-packages\torch\_dynamo\convert_frame.py:2287] due to: 
W0921 07:22:38.414000 19272 Lib\site-packages\torch\_dynamo\convert_frame.py:2287] Traceback (most recent call last):
W0921 07:22:38.414000 19272 Lib\site-packages\torch\_dynamo\convert_frame.py:2287]   File "f:\paul\structured-llm-notebooks\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 2192, in __call__
W0921 07:22:38.414000 19272 Lib\site-packages\torch\_dynamo\convert_frame.py:2287]     result = self._inner_convert(
W0921 07:22:38.414000 19272 Lib\site-packages\torch\_dynamo\convert_frame.py:2287]              ^^^^^^^^^^^^^^^^^^^^
W0921 07:22:38.414000 19272 Lib\site-packages\torch\_dynamo\convert_frame.py:2287]   File "f:\paul\structured-llm-notebooks\.venv\Lib\

Generated expression: 1.1059463887374257*1
✅ Always syntactically valid arithmetic


## 3. SQL Grammar — Generating Valid SELECT/WHERE Queries

In [4]:
simple_sql_grammar = r"""
    start: "SELECT " columns " FROM " table_name
    columns: "*" | column ("," column)*
    column: /[a-z_]+/
    table_name: /[a-z_]+/
"""

result = model("Generate a SQL query to get all users:", CFG(simple_sql_grammar))

print(f"SQL: {result}")
print("✅ Always syntactically valid SQL")

SQL: SELECT * FROM users
✅ Always syntactically valid SQL


In [5]:
# Extended SQL with WHERE clause
sql_with_where = r"""
    start: "SELECT " columns " FROM " table_name (" WHERE " condition)?
    columns: "*" | column ("," column)*
    column: /[a-z_]+/
    table_name: /[a-z_]+/
    condition: column "=" value
             | column ">" value
             | column "<" value
    value: /[0-9]+/ | /'[^']*'/
"""

result = model("Generate SQL to find users older than 18:", CFG(sql_with_where))

print(f"SQL with WHERE: {result}")

SQL with WHERE: SELECT * FROM users WHERE age>18


## 4. Python Expression Grammar

In [6]:
python_expr_grammar = r"""
start: NUMBER (op NUMBER)+
op: "+" | "-" | "*" | "/" | "**"
NUMBER: /[0-9]+(\.[0-9]+)?/
"""

result = model(
    "Compute three plus four, as a plain arithmetic expression:",
    CFG(python_expr_grammar),
)

print(f"Python expression: {result}")

Python expression: 3+4.0


## 5. Custom DSL Grammar — Define Your Own Language

In [7]:
# Define a simple query DSL
query_dsl = r"""
    start: query
    query: "FIND " entity (" WHERE " condition)?
    entity: "users" | "products" | "orders"
    condition: field "=" value
    field: "name" | "status" | "price" | "date"
    value: /[a-zA-Z0-9_]+/
"""

result = model("Write a query to find products where status=active:", CFG(query_dsl))

print(f"DSL query: {result}")

DSL query: FIND products WHERE status=ACTIVE


## 6. CFG vs Raw Generation — Syntax Error Rate Comparison

In [ ]:
# Benchmark: generate 10 SQL queries with and without CFG
prompts = [
    "Get all users",
    "Find orders where total > 100",
    "Select products with status active",
    "List customers from New York",
    "Count orders by status",
]

print("Without CFG (Raw):")
for prompt in prompts:
    # Notice the CFG parameter is removed here
    result = model(f"{prompt}:") 
    print(f"  {result}")
print("="*60)
print("\nWith CFG (Outlines):")
for prompt in prompts:
    result = model(f"{prompt}:", CFG(simple_sql_grammar))
    print(f"  {result}")

print("\n✅ CFG guarantees: 100% syntactically valid output")
print("   Raw prompting: often produces invalid SQL with extra text")

Without CFG (Raw):
  I'm sorry, but I need more context to provide the information you're looking for. Could you
  I'm sorry, but I don't have the ability to directly process or execute code. However,
  Sure! To provide you with the latest list of products with an "active" status from Alibaba Cloud
  As an AI language model, I don't have access to real-time data or specific information about customer
  To count the number of orders by their status in Alibaba Cloud, you can use various methods depending on
With CFG (Outlines):
  SELECT * FROM users
  SELECT * FROM your_table_name_where_total_greater_than_threshold_value
  SELECT product_name FROM orders_table_warehouse_product_table_product_status_active
  SELECT customer_name FROM customers_table_abc__table__where_city_is_new_york
  SELECT status,_quantity,total_price FROM orders_table

✅ CFG guarantees: 100% syntactically valid output
   Raw prompting: often produces invalid SQL with extra text


## 7. vLLM Integration for Production Serving

In [ ]:
# When serving with vLLM, use the guided_* parameters
# These use Outlines under the hood in vLLM V0

vllm_example = """
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

completion = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B",
    messages=[{"role": "user", "content": "Generate SQL"}],
    extra_body={
        "guided_grammar": simple_sql_grammar
        # OR: "guided_json": schema
        # OR: "guided_regex": r"..."
        # OR: "guided_choice": ["yes", "no"]
    }
)
"""

print(vllm_example)
print(
    "\n💡 Note: vLLM V1 defaults to xgrammar. Use --guided-decoding-backend outlines for full Outlines support."
)


from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

completion = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B",
    messages=[{"role": "user", "content": "Generate SQL"}],
    extra_body={
        "guided_grammar": simple_sql_grammar
        # OR: "guided_json": schema
        # OR: "guided_regex": r"..."
        # OR: "guided_choice": ["yes", "no"]
    }
)


💡 Note: vLLM V1 defaults to xgrammar. Use --guided-decoding-backend outlines for full Outlines support.


: 